# 1 amending src files

In [ ]:
FDIR = 'modf_rntg'

In [45]:
import pandas as pd

def ld(p):
    return pd.read_csv(p, sep=',')

In [46]:
def rm_col(ncols, c):
    if c in ncols:
        ncols.remove(c)
    return ncols

In [47]:
IOU_COLS_GDA = ["3_test_GDA_iou", "3_test/GDA/iou", "1_test/GDA/iou",  
                "2_test/GDA/iou", "1_test_GDA_iou", "2_test_GDA_iou"]
IOU_COLS_SYNT = ["3_test_SYNT_iou", "1_test/SYNT/iou", "2_test/SYNT/iou", 
            "3_test/SYNT/iou", "1_test_SYNT_iou", "2_test_SYNT_iou"]
IOU_COLS_DK = ["3_test_DK_iou", "1_test/DK/iou", "2_test/DK/iou", 
            "3_test/DK/iou", "1_test_DK_iou", "2_test_DK_iou"]

IOU_COLS = IOU_COLS_GDA + IOU_COLS_SYNT + IOU_COLS_DK
# SPLIT_COLUMNS = ['val', 'Sweep']
SCOLS = ["batch_size", "epochs", "warmup_epochs", "wd", "ema", "sub", 
        "ID", "loss", "lrdec", "lrenc", "mod_ph1", "mod_ph2", "Sweep",
        "val", "1_epoch", "2_epoch", "3_epoch", 'Runtime'] + IOU_COLS

def ld_scols(p):
    df = pd.read_csv(p, sep=',')
    cols = df.columns
    cols = list(set(cols) & set(SCOLS))
    ncols = list(set(cols) & set(SCOLS))
    df = df[df['State'] == 'finished']
    # if 'ID' in cols:
    #     df.set_index('ID', inplace=True, drop=False)
        # df.reindex('ID')
    if 'ph3' in p:
        ncols = rm_col(ncols, "2_test/DK/iou")
        ncols = rm_col(ncols, "2_test/SYNT/iou")
        ncols = rm_col(ncols, "2_test/GDA/iou")
        ncols = rm_col(ncols, "2_test_DK_iou")
        ncols = rm_col(ncols, "2_test_GDA_iou")
        ncols = rm_col(ncols, "2_test_SYNT_iou")
    if 'S' in p:
        ncols = rm_col(ncols, "1_test_DK_iou")
        ncols = rm_col(ncols, "1_test_GDA_iou")
        ncols = rm_col(ncols, "1_test_SYNT_iou")
    for col in cols:
        if '/' in col:
            # print('sign found in', col)
            df[col.replace('/', '_')] = df[col]
            df = df.drop(columns=[col])
            if col in ncols:
                ncols.remove(col)
            ncols.append(col.replace('/', '_'))
    # print(ncols)
    return df[ncols].fillna({'ema': False})

In [48]:
def map_dict_val(d: dict, val, deff):
    for k, v in d.items():
        # print(k, v)
        if k in val:
            # print(len(val.split('_')))
            if ('_' in val and len(val.split('_')) > 3) or 'ph1_gda' in val:
                return 'gda'
            return v
    if 'ph1_gda' in val:
        return 'gda'
    return deff

In [49]:
fn_tr = {
    'ph2': 'subm',
    'ph3': 'dk',
    'dk': 'dk',
    'subs': 'sub',
    'S': 'sub',
}

def add_tr(df, fn):
    df['train'] = map_dict_val(fn_tr, fn, 's')
    return df

In [50]:
from numpy import select

val_val = {
    "/users/project1/pt01299/synt/gda70/train/index_val.csv": "gda",
    "/users/project1/pt01299/synt/gda70/train/index_val_val.csv": "gda",
    "/users/project1/pt01299/synt/gda70/train/index_val_tr.csv": "gda",
    "/users/project1/pt01299/synt/segformer_dataset255_all/val/index.csv": "s",
    "/users/project1/pt01299/synt/mix_val.csv": "m",
    "/users/project1/pt01299/synt/DK/osfstorage/dataset_v2/solardk_dataset_neurips_v2/gentofte_trainval/val/index.csv": "dk"
}

def map_val(df):
    if 'val' not in df.columns:
        df['val'] = list(val_val.keys())[0]
    v = df['val'].astype(str)
    conds = [v.str.contains(k, regex=False, na=False) for k in val_val.keys()]
    choices = list(val_val.values())
    df['val'] = select(conds, choices, default='s')
    return df

In [51]:
def add_tr_val(df):
    df['tr_val'] = df['train'] + '_' + df['val']
    return df

In [52]:
def add_src(df, fn):
    sp = fn.split('_')
    if len(sp) > 2:
        df['src'] = sp[1] + '_' + sp[2].replace('.csv', '')
    elif 'm' in fn:
        df['src'] = 'subm'
    else:
        df['src'] = 'x'
    df['fn'] = fn
    return df

In [53]:
f = 'dk.csv'
f = 'subs_m.csv'
f = 'ph2_dk_dk.csv'
f = 'ph1_gda.csv'
df = ld_scols(f)
df = add_tr(df, f)
df = map_val(df)
df = add_tr_val(df)
df = add_src(df, f)
if 'ID' not in df.columns:
        df['ID'] = 'keine'
df.head()

,batch_size,mod_ph1,sub,mod_ph2,lrdec,epochs,ema,warmup_epochs,lrenc,loss,...,1_test_SYNT_iou,1_test_GDA_iou,3_test_SYNT_iou,3_test_GDA_iou,3_test_DK_iou,train,tr_val,src,fn,ID
0,2,NaN,NaN,NaN,0.000005,32,False,4,0.00005,Dice+Focal,...,0.498818,0.726285,NaN,NaN,NaN,gda,gda_dk,x,ph1_gda.csv,keine
1,2,NaN,NaN,NaN,0.000005,32,False,0,0.00005,Dice+Focal,...,0.423319,0.724569,NaN,NaN,NaN,gda,gda_dk,x,ph1_gda.csv,keine
2,8,NaN,NaN,NaN,0.000005,32,False,0,0.00005,Dice+Focal,...,0.462638,0.722245,NaN,NaN,NaN,gda,gda_dk,x,ph1_gda.csv,keine
3,4,NaN,NaN,NaN,0.000005,32,False,4,0.00005,Dice+Focal,...,0.431171,0.716016,NaN,NaN,NaN,gda,gda_dk,x,ph1_gda.csv,keine
4,4,NaN,NaN,NaN,0.000005,32,False,0,0.00005,Dice+Focal,...,0.397487,0.715953,NaN,NaN,NaN,gda,gda_gda,x,ph1_gda.csv,keine


In [ ]:
def proc_f(f):
    df = ld_scols(f)
    df = add_tr(df, f)
    df = map_val(df)
    df = add_tr_val(df)
    df = add_src(df, f)
    if 'ID' not in df.columns:
        df['ID'] = 'keine'
        print('no id in', f)
    df.to_csv(f'{FDIR}/{f}')
    
    return df

In [55]:
import os

files = [f for f in os.listdir() if f.lower().endswith(".csv")]
files.remove('da.csv')
files

['30.csv',
 'dk.csv',
 'full.csv',
 'ph1_gda.csv',
 'ph2_dk_dk.csv',
 'ph2_dk_gda.csv',
 'ph2_sub_gda.csv',
 'ph2_sub_s.csv',
 'ph2_s_gda.csv',
 'ph2_s_s.csv',
 'ph3_subm_gda.csv',
 'ph3_subm_gda_gda.csv',
 'ph3_subm_m.csv',
 'ph3_subm_m_gda.csv',
 'ph3_subm_s.csv',
 'ph3_subm_s_gda.csv',
 'S3.csv',
 'S4.csv',
 'short.csv',
 'subs_16.csv',
 'subs_16A.csv',
 'subs_m.csv']

## proces each file

In [ ]:
ph1 = pd.DataFrame()
ph2 = pd.DataFrame()
ph3 = pd.DataFrame()

for ff in files:
    print(ff)
    df = proc_f(ff)
    if 'ph2' in ff:
        ph2 = pd.concat([df, ph2], ignore_index=True)
    elif 'ph3' in ff:
        ph3 = pd.concat([df, ph3], ignore_index=True)
    else:
        try:
            if 'S' in ff:
                print(len(df.columns))
                print(len(pd.unique(df.columns)))
                print(df.columns)
            ph1 = pd.concat([df, ph1], ignore_index=True)
        except Exception as e:
            print('ph1', ff, e)

# ph1 = ph1.dropna(axis='columns')
ph2 = ph2.dropna(axis='columns')
ph3 = ph3.dropna(axis='columns')

ph1.to_csv(f'{FDIR}/proc_ph1.csv')
ph2.to_csv(f'{FDIR}/proc_ph2.csv')
ph3.to_csv(f'{FDIR}/proc_ph3.csv')

30.csv
dk.csv
full.csv
ph1_gda.csv
no id in ph1_gda.csv
ph2_dk_dk.csv
ph2_dk_gda.csv
ph2_sub_gda.csv
ph2_sub_s.csv
ph2_s_gda.csv


C:\Users\admin\AppData\Local\Temp\ipykernel_16944\2474272437.py:43: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df[ncols].fillna({'ema': False})
C:\Users\admin\AppData\Local\Temp\ipykernel_16944\2474272437.py:43: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df[ncols].fillna({'ema': False})


ph2_s_s.csv
ph3_subm_gda.csv
ph3_subm_gda_gda.csv
no id in ph3_subm_gda_gda.csv
ph3_subm_m.csv
ph3_subm_m_gda.csv
no id in ph3_subm_m_gda.csv
ph3_subm_s.csv
ph3_subm_s_gda.csv
no id in ph3_subm_s_gda.csv
S3.csv
20
20
Index(['batch_size', 'ID', 'sub', 'lrdec', 'epochs', 'warmup_epochs', 'lrenc',
       'loss', 'Sweep', 'val', 'Runtime', 'wd', '1_epoch', '1_test_DK_iou',
       '1_test_SYNT_iou', '1_test_GDA_iou', 'train', 'tr_val', 'src', 'fn'],
      dtype='object')
S4.csv
18
18
Index(['loss', 'Runtime', 'wd', 'lrdec', 'batch_size', '1_epoch', 'ID',
       'warmup_epochs', 'lrenc', 'sub', '1_test_SYNT_iou', '1_test_GDA_iou',
       '1_test_DK_iou', 'train', 'val', 'tr_val', 'src', 'fn'],
      dtype='object')
short.csv
subs_16.csv
subs_16A.csv
subs_m.csv
no id in subs_m.csv


C:\Users\admin\AppData\Local\Temp\ipykernel_16944\2474272437.py:43: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df[ncols].fillna({'ema': False})
C:\Users\admin\AppData\Local\Temp\ipykernel_16944\2474272437.py:43: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df[ncols].fillna({'ema': False})


drop duplicate iou cols
drop empty ious
====stack all
====join 1-2 by mod1
====join 2-3 by mod3
====generate comb_key 1-2-3
====grouped agg stats by comb_key
stack disjoint to joint - to załatwia skrypt etl, tutaj tylko prawdziwe i pełne fazy

# 2 joins + save wide

In [57]:
cnc = pd.concat([ph1, ph2, ph3])
cnc.head()

,loss,Sweep,val,Runtime,wd,lrdec,batch_size,1_epoch,epochs,warmup_epochs,...,ID,mod_ph2,3_epoch,3_test_SYNT_iou,3_test_GDA_iou,3_test_DK_iou,2_epoch,2_test_DK_iou,2_test_SYNT_iou,2_test_GDA_iou
0,Dice+Focal,8ej2yv2y,m,4416,0.05,0.000008,8,19.0,20.0,2,...,keine,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Dice+Focal,8ej2yv2y,s,4085,0.01,0.000008,8,19.0,20.0,2,...,keine,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Dice+Focal,8ej2yv2y,m,5381,0.03,0.000008,16,19.0,20.0,2,...,keine,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Dice+Focal,8ej2yv2y,gda,4572,0.03,0.000008,16,19.0,20.0,4,...,keine,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Dice+Focal,8ej2yv2y,gda,2996,0.05,0.000008,16,19.0,20.0,2,...,keine,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
cnc.to_csv(f'{FDIR}/cnc123b.csv')

In [59]:
df12 = pd.merge(ph1, ph2, how='inner', left_on=['ID'], right_on=['mod_ph1'])
df12.head()

,loss_x,Sweep_x,val_x,Runtime_x,wd_x,lrdec_x,batch_size_x,1_epoch,epochs_x,warmup_epochs_x,...,lrenc_y,ema_y,sub_y,2_test_DK_iou,2_test_SYNT_iou,2_test_GDA_iou,train_y,tr_val_y,src_y,fn_y
0,Dice+Focal,7l3d9j8z,gda,979,0.05,0.000008,16,15.0,16.0,6,...,0.000010,False,15,0.789495,0.788783,0.739968,subm,subm_gda,sub_gda,ph2_sub_gda.csv
1,Dice+Focal,7l3d9j8z,gda,979,0.05,0.000008,16,15.0,16.0,6,...,0.000010,False,composite,0.808589,0.789358,0.736804,subm,subm_s,sub_gda,ph2_sub_gda.csv
2,Dice+Focal,7l3d9j8z,gda,979,0.05,0.000008,16,15.0,16.0,6,...,0.000005,False,15,0.788369,0.791244,0.736472,subm,subm_gda,sub_gda,ph2_sub_gda.csv
3,Dice+Focal,7l3d9j8z,gda,979,0.05,0.000008,16,15.0,16.0,6,...,0.000010,False,mix,0.801673,0.783520,0.734996,subm,subm_m,sub_gda,ph2_sub_gda.csv
4,Dice+Focal,7l3d9j8z,gda,979,0.05,0.000008,16,15.0,16.0,6,...,0.000010,False,15,0.803570,0.799280,0.734078,subm,subm_gda,sub_gda,ph2_sub_gda.csv


In [60]:
df123 = pd.merge(df12, ph3, how='inner', left_on=['ID_y'], right_on=['mod_ph2'])
df123.head()

,loss_x,Sweep_x,val_x,Runtime_x,wd_x,lrdec_x,batch_size_x,1_epoch,epochs_x,warmup_epochs_x,...,wd,3_epoch_y,3_test_SYNT_iou_y,3_test_GDA_iou_y,3_test_DK_iou_y,train,tr_val,src,fn,ID
0,Dice+Focal,7l3d9j8z,gda,632,0.05,0.000008,8,15.0,16.0,6,...,0.015,7.0,0.784910,0.750969,0.796645,gda,gda_dk,subm_m,ph3_subm_m_gda.csv,keine
1,Dice+Focal,7l3d9j8z,gda,632,0.05,0.000008,8,15.0,16.0,6,...,0.015,7.0,0.794183,0.741155,0.805439,gda,gda_dk,subm_m,ph3_subm_m_gda.csv,keine
2,Dice+Focal,7l3d9j8z,gda,632,0.05,0.000008,8,15.0,16.0,6,...,0.015,3.0,0.773208,0.729857,0.797129,gda,gda_dk,subm_m,ph3_subm_m_gda.csv,keine
3,Dice+Focal,7l3d9j8z,gda,632,0.05,0.000008,8,15.0,16.0,6,...,0.030,11.0,0.792205,0.750134,0.811760,dk,dk_gda,subm_m,ph3_subm_m.csv,xkqymss2
4,Dice+Focal,7l3d9j8z,gda,632,0.05,0.000008,8,15.0,16.0,6,...,0.050,5.0,0.803058,0.748709,0.813642,dk,dk_gda,subm_m,ph3_subm_m.csv,mhg6zpu0


In [61]:
df = df123
df['comb_key'] = df['tr_val_x'] + '|' + \
                df['tr_val_y'] + '|' + \
                df['tr_val']
df.head()

,loss_x,Sweep_x,val_x,Runtime_x,wd_x,lrdec_x,batch_size_x,1_epoch,epochs_x,warmup_epochs_x,...,3_epoch_y,3_test_SYNT_iou_y,3_test_GDA_iou_y,3_test_DK_iou_y,train,tr_val,src,fn,ID,comb_key
0,Dice+Focal,7l3d9j8z,gda,632,0.05,0.000008,8,15.0,16.0,6,...,7.0,0.784910,0.750969,0.796645,gda,gda_dk,subm_m,ph3_subm_m_gda.csv,keine,sub_gda|subm_m|gda_dk
1,Dice+Focal,7l3d9j8z,gda,632,0.05,0.000008,8,15.0,16.0,6,...,7.0,0.794183,0.741155,0.805439,gda,gda_dk,subm_m,ph3_subm_m_gda.csv,keine,sub_gda|subm_m|gda_dk
2,Dice+Focal,7l3d9j8z,gda,632,0.05,0.000008,8,15.0,16.0,6,...,3.0,0.773208,0.729857,0.797129,gda,gda_dk,subm_m,ph3_subm_m_gda.csv,keine,sub_gda|subm_m|gda_dk
3,Dice+Focal,7l3d9j8z,gda,632,0.05,0.000008,8,15.0,16.0,6,...,11.0,0.792205,0.750134,0.811760,dk,dk_gda,subm_m,ph3_subm_m.csv,xkqymss2,sub_gda|subm_m|dk_gda
4,Dice+Focal,7l3d9j8z,gda,632,0.05,0.000008,8,15.0,16.0,6,...,5.0,0.803058,0.748709,0.813642,dk,dk_gda,subm_m,ph3_subm_m.csv,mhg6zpu0,sub_gda|subm_m|dk_gda


In [ ]:
df.to_csv(f'{FDIR}/ph123b.csv') # OK